In [54]:
import pandas as pd
df = pd.read_csv('jobs.csv') 

print("Dataset Shape:", df.shape)
print("\nColumns in Dataset:")
print(df.columns.tolist())
print("\nFirst row sample:")
display(df.head(1))

Dataset Shape: (660, 27)

Columns in Dataset:
['Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'min_salary', 'max_salary', 'avg_salary', 'job_state', 'same_state', 'company_age', 'python', 'excel', 'hadoop', 'spark', 'aws', 'tableau', 'big_data', 'job_simp', 'seniority']

First row sample:


,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Type of ownership,Industry,...,company_age,python,excel,hadoop,spark,aws,tableau,big_data,job_simp,seniority
0,Sr Data Scientist,137-171,Description\n\nThe Senior Data Scientist is re...,3.1,Healthfirst,"New York, NY","New York, NY",1001 to 5000 employees,Nonprofit Organization,Insurance Carriers,...,27,0,0,0,0,1,0,0,data scientist,senior


In [55]:
#Relevant columns
cols = [
    'Job Title', 'job_simp', 'seniority', 'Company Name', 
    'Location', 'job_state', 'Rating', 'avg_salary', 
    'Sector', 'Job Description',
    'python', 'excel', 'hadoop', 'spark', 'aws', 'tableau', 'big_data'
]

jobs_df=df[cols].copy().dropna(subset=['Job Title','Job Description']).reset_index(drop=True)

skill_cols = ['python', 'excel', 'hadoop', 'spark', 'aws', 'tableau', 'big_data']

def extract_skills(row):
    active_skills = []
    for skill in skill_cols:
        if row[skill] == 1:
            active_skills.append(skill)
    return " ".join(active_skills)  # <-- Aligned with 'for' (outside the loop)


jobs_df['skills_extracted'] = jobs_df.apply(extract_skills, axis=1)


jobs_df[['Job Title', 'skills_extracted']].head()


,Job Title,skills_extracted
0,Sr Data Scientist,aws
1,Data Scientist,hadoop big_data
2,Data Scientist,python excel aws
3,Data Scientist,python excel aws
4,Data Scientist,python excel


In [56]:
import re

def clean_text(text):
    # 1. Convert to string and make everything lowercase
    text = str(text).lower()
    
    # 2. Replace line breaks (\n, \r) and tabs (\t) with a single space
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # 3. Remove punctuation and special symbols (keep only letters, numbers, and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # 4. Remove extra spaces and strip leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply this function to clean the Job Description column
jobs_df['cleaned_desc'] = jobs_df['Job Description'].apply(clean_text)

# Preview the original vs the cleaned version for the first job
print("--- ORIGINAL DESCRIPTION (First 150 chars) ---")
print(jobs_df['Job Description'].iloc[0][:150])

print("\n--- CLEANED DESCRIPTION (First 150 chars) ---")
print(jobs_df['cleaned_desc'].iloc[0][:150])

--- ORIGINAL DESCRIPTION (First 150 chars) ---
Description

The Senior Data Scientist is responsible for defining, building, and improving statistical models to improve business processes and outco

--- CLEANED DESCRIPTION (First 150 chars) ---
description the senior data scientist is responsible for defining building and improving statistical models to improve business processes and outcomes


In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(max_features=4000, stop_words='english',ngram_range=(1,2))
tfidf_matrix=tfidf.fit_transform(jobs_df['Job Description'])
print(f"TF-IDF Matrix successfully generated!")
print(f"Matrix Shape:{tfidf_matrix.shape}")

TF-IDF Matrix successfully generated!
Matrix Shape:(660, 4000)


In [58]:
# Looking at 10 random vocabulary features picked by the model
feature_names = tfidf.get_feature_names_out()
print("Total features:", len(feature_names))
print("Sample features:", feature_names[200:215])

Total features: 4000
Sample features: ['analytics machine' 'analytics platform' 'analytics platforms'
 'analytics projects' 'analytics solutions' 'analytics support'
 'analytics team' 'analytics tools' 'analyze' 'analyze data'
 'analyze large' 'analyzes' 'analyzing' 'analyzing data' 'ancestry']


In [68]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def calibrate_score(raw_score, benchmark=0.25):
    """
    Transforms raw cosine similarity (0.0 to ~0.25) into an intuitive percentage (0% to 100%).
    """
    if raw_score <= 0:
        return 0.0
    
    # Scale relative to the benchmark ceiling
    calibrated = (raw_score / benchmark) * 100
    
    # Cap at 99.0% so only an exact duplicate gets 100%
    return round(min(calibrated, 99.0), 1)


def match_resume(resume_text, top_n=5):
    # 1. Clean the user's input text
    cleaned_input = clean_text(resume_text)
    
    # 2. Vectorize the input
    user_vector = tfidf.transform([cleaned_input])
    
    # 3. Compute cosine similarity
    raw_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()
    
    # 4. Pair index with score using a standard loop
    scored_jobs = []
    for index, score in enumerate(raw_scores):
        scored_jobs.append((index, score))
        
    # 5. Sort descending by score
    sorted_jobs = sorted(scored_jobs, key=lambda x: x[1], reverse=True)
    top_matches = sorted_jobs[:top_n]
    
    print(f"=== TOP {top_n} MATCHING ROLES (CALIBRATED) ===\n")
    
    # 6. Extract candidate's skills using a standard loop
    tracked_skills = ['python', 'excel', 'hadoop', 'spark', 'aws', 'tableau', 'big_data']
    user_skills_found = []
    
    for skill in tracked_skills:
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, cleaned_input):
            user_skills_found.append(skill)
            
    print(f"Skills detected in your input: {', '.join(user_skills_found) if user_skills_found else 'None'}\n")
    print("=" * 60)
    
    # 7. Print matches and analyze skill gaps using standard loops
    for rank, (job_index, raw_score) in enumerate(top_matches, start=1):
        job_row = jobs_df.iloc[job_index]
        display_score = calibrate_score(raw_score)
        
        # Split job skills into a list
        job_skills = job_row['skills_extracted'].split()
        
        # Standard loops for matched and missing skills
        matched_skills = []
        missing_skills = []
        
        for s in job_skills:
            if s in user_skills_found:
                matched_skills.append(s)
            else:
                missing_skills.append(s)
        
        print(f"Rank {rank}: {job_row['Job Title']} at {job_row['Company Name']}")
        print(f"  - Match Confidence: {display_score}% (Raw Cosine: {round(raw_score, 3)})")
        print(f"  - Location: {job_row['Location']}")
        print(f"  - Salary: ${job_row['avg_salary']}K/yr" if job_row['avg_salary'] > 0 else "  - Salary: Not Disclosed")
        print(f"  - Matched Skills: {', '.join(matched_skills) if matched_skills else 'General Text Overlap'}")
        if missing_skills:
            print(f"  - Missing Skills to Learn: {', '.join(missing_skills)}")
        print("-" * 60)

In [69]:
sample_resume = """
Data analyst with hands-on experience in Python, SQL, and Excel. 
Skilled in building dashboards with Tableau, data wrangling using Pandas, 
and basic cloud computing using AWS. Strong background in statistical analysis.
"""

match_resume(sample_resume, top_n=5)

=== TOP 5 MATCHING ROLES (CALIBRATED) ===

Skills detected in your input: python, excel, aws, tableau

Rank 1: Data Scientist at TechProjects
  - Match Confidence: 76.7% (Raw Cosine: 0.192)
  - Location: New York, NY
  - Salary: $113K/yr
  - Matched Skills: python, excel, tableau
------------------------------------------------------------
Rank 2: Data Scientist at TechProjects
  - Match Confidence: 76.7% (Raw Cosine: 0.192)
  - Location: New York, NY
  - Salary: $92K/yr
  - Matched Skills: python, excel, tableau
------------------------------------------------------------
Rank 3: Data Analyst at Envision LLC
  - Match Confidence: 65.6% (Raw Cosine: 0.164)
  - Location: San Francisco, CA
  - Salary: $99K/yr
  - Matched Skills: python, tableau
  - Missing Skills to Learn: hadoop, spark, big_data
------------------------------------------------------------
Rank 4: Sr. Data Analyst at Community Behavioral Health
  - Match Confidence: 63.9% (Raw Cosine: 0.16)
  - Location: Philadelphia, PA

In [71]:
import pickle
import os

with open('jobs_df.pkl', 'wb') as file:
    pickle.dump(jobs_df, file)

with open('tfidf_vectorizer.pkl', 'wb') as file:
    pickle.dump(tfidf, file)

with open('tfidf_matrix.pkl', 'wb') as file:
    pickle.dump(tfidf_matrix, file)

print("All 3 artifacts successfully serialized and saved to disk!")

All 3 artifacts successfully serialized and saved to disk!
